# ChronicCare Nour — Risk Stratification with Explainable Boosting Machine (EBM)

Trains an **EBM multiclass classifier** that categorises diabetic patients into
**low / moderate / high** risk based on patient profile, chronic context, real-time
sensor vitals, and Gemini-extracted symptoms.

| Group | Feature | Description |
|---|---|---|
| Profile | `age` | Patient age |
| Profile | `gender` | 0 = M, 1 = F |
| Profile | `bmi` | Body Mass Index |
| Context | `hba1c` | Last recorded HbA1c (%) |
| Context | `has_hypertension` | Comorbidity flag |
| Context | `has_heart_disease` | Comorbidity flag |
| Context | `medication_count` | Number of active medications |
| Vitals | `glucose` | Real-time glucose (mg/dL) |
| Vitals | `hr` | Heart rate (bpm) |
| Vitals | `spo2` | Blood oxygen saturation (%) |
| Vitals | `steps` | Daily step count |
| Vitals | `sleep_hours` | Hours of sleep |
| Symptoms | `confusion` | Gemini-extracted flag |
| Symptoms | `tremors` | Gemini-extracted flag |
| Symptoms | `thirst` | Gemini-extracted flag |

In [ ]:
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from interpret.glassbox import ExplainableBoostingClassifier
from sklearn.metrics import ConfusionMatrixDisplay, classification_report
from sklearn.model_selection import train_test_split

FEATURE_NAMES = [
    "age",
    "gender",           # 0 = M, 1 = F
    "bmi",
    "hba1c",
    "has_hypertension",
    "has_heart_disease",
    "medication_count",
    "glucose",
    "hr",
    "spo2",
    "steps",
    "sleep_hours",
    "confusion",
    "tremors",
    "thirst",
]
RISK_LABELS = ["low", "moderate", "high"]
MODEL_PATH = Path("../models/risk_ebm.pkl")

## 1. Generate Enriched Synthetic Dataset

Each sample is drawn with:
- **Profile**: age, gender, BMI (correlated with age)
- **Context**: HbA1c, comorbidities and medication count (correlated with age/BMI)
- **Vitals**: glucose, HR, SpO2, steps, sleep — driven by the assigned risk state
- **Symptoms**: confusion / tremors / thirst — derived from glucose thresholds 

In [ ]:
def generate_dataset(n_samples: int = 2000) -> pd.DataFrame:
    rng = np.random.default_rng(42)
    rows = []

    for _ in range(n_samples):
        # Profile
        age    = int(rng.integers(20, 85))
        gender = int(rng.integers(0, 2))          # 0 = M, 1 = F
        bmi    = float(np.clip(rng.normal(27.0 + age * 0.05, 5.0), 17.0, 50.0))

        # Chronic context
        hba1c             = float(rng.uniform(5.0, 12.0))
        has_hypertension  = int(rng.random() < (0.1 + age * 0.005 + (bmi - 25) * 0.01))
        has_heart_disease = int(rng.random() < (0.05 + age * 0.003))
        medication_count  = int(np.clip(rng.integers(0, 3) + has_hypertension + has_heart_disease, 0, 6))

        # Lifestyle
        sleep_hours  = float(rng.uniform(4.0, 9.0))
        sleep_impact = (8.0 - sleep_hours) * 10.0 if sleep_hours < 7.0 else 0.0
        steps        = int(rng.integers(500, 15000))
        step_impact  = (steps / 1000.0) * -5.0

        # Risk state — comorbidities shift probability toward higher risk
        extra_high = 0.05 * (has_hypertension + has_heart_disease)
        p_low  = max(0.10, 0.50 - extra_high)
        p_high = min(0.40, 0.20 + extra_high)
        p_mod  = round(1.0 - p_low - p_high, 6)
        target_state = int(rng.choice([0, 1, 2], p=[p_low, p_mod, p_high]))

        if target_state == 0:   # LOW risk
            glucose = float(rng.uniform(90.0, 140.0)) + sleep_impact + step_impact
            hr      = int(rng.integers(60, 85))
            spo2    = int(rng.integers(96, 100))
        elif target_state == 1: # MODERATE risk
            glucose = float(rng.uniform(160.0, 240.0)) + sleep_impact
            hr      = int(rng.integers(80, 100))
            spo2    = int(rng.integers(94, 98))
        else:                   # HIGH risk — hypoglycaemia or hyperglycaemic emergency
            is_hypo = rng.random() < 0.3
            if is_hypo:
                glucose = float(rng.uniform(40.0, 65.0))
                hr      = int(rng.integers(105, 130))  # tachycardia during hypo
            else:
                glucose = float(rng.uniform(260.0, 450.0))
                hr      = int(rng.integers(90, 110))
            spo2 = int(rng.integers(90, 95))

        glucose = max(40.0, glucose)

        # Symptom flags — mocking Gemini extraction output
        confusion = int(target_state == 2 and rng.random() < 0.7)
        tremors   = int(glucose < 70.0 and rng.random() < 0.8)
        thirst    = int(glucose > 200.0 and rng.random() < 0.6)

        rows.append({
            "age": age, "gender": gender, "bmi": bmi,
            "hba1c": hba1c, "has_hypertension": has_hypertension,
            "has_heart_disease": has_heart_disease, "medication_count": medication_count,
            "glucose": glucose, "hr": hr, "spo2": spo2,
            "steps": steps, "sleep_hours": sleep_hours,
            "confusion": confusion, "tremors": tremors, "thirst": thirst,
            "risk_state": target_state,
        })

    return pd.DataFrame(rows)


df = generate_dataset(2000)
df["risk_label"] = df["risk_state"].map({0: "low", 1: "moderate", 2: "high"})
print(df["risk_label"].value_counts())
df.head()

## 2. Train EBM Multiclass Classifier

In [ ]:
X = df[FEATURE_NAMES].to_numpy(dtype=float)
y = df["risk_state"].to_numpy()

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

ebm = ExplainableBoostingClassifier(
    feature_names=FEATURE_NAMES,
    n_jobs=-1,
    random_state=42,
)
ebm.fit(X_train, y_train)

y_pred = ebm.predict(X_test)
print(classification_report(y_test, y_pred, target_names=RISK_LABELS))

## 3. Confusion Matrix

In [ ]:
fig, ax = plt.subplots(figsize=(6, 5))
ConfusionMatrixDisplay.from_predictions(
    y_test, y_pred, display_labels=RISK_LABELS, ax=ax
)
ax.set_title("EBM Risk Stratification — Confusion Matrix")
plt.tight_layout()
plt.show()

## 4. Global Feature Importances

EBM computes the **mean absolute contribution** of each feature across all training samples.

In [ ]:
importances = ebm.term_importances()
order = np.argsort(importances)[::-1]

fig, ax = plt.subplots(figsize=(10, 4))
ax.bar([FEATURE_NAMES[i] for i in order], importances[order])
ax.set_xlabel("Feature")
ax.set_ylabel("Mean |contribution|")
ax.set_title("EBM — Global Feature Importances")
plt.xticks(rotation=35, ha="right")
plt.tight_layout()
plt.show()

## 5. EBM Shape Functions

Shape functions show *how* each feature value pushes the prediction score up or down for each class.
This is the explainability advantage of EBM over a black-box model.

In [ ]:
ebm_global = ebm.explain_global(name="EBM Global")
# Render interactive dashboard (works in Jupyter)
from interpret import show
show(ebm_global)

## 6. Per-Instance Local Explanation

For a single patient, EBM tells you **which features drove this specific prediction** and by how much.

In [ ]:
# Pick a high-risk test sample to explain
idx = next(i for i, lbl in enumerate(y_test) if lbl == 2)
X_sample = X_test[idx : idx + 1]

ebm_local = ebm.explain_local(X_sample, name="EBM Local")
show(ebm_local)

## 7. Save Model

In [ ]:
MODEL_PATH.parent.mkdir(parents=True, exist_ok=True)
joblib.dump(ebm, MODEL_PATH)
print(f"Saved → {MODEL_PATH.resolve()}")

## 8. Inference Demo

In [ ]:
import sys
sys.path.insert(0, str(Path("../")))

from src.domain.models.risk_models import RiskFeatures
from src.infrastructure.ml.risk_classifier import RiskClassifier

rc = RiskClassifier(model_path=MODEL_PATH)

patients = [
    RiskFeatures(
        age=35, gender="F", bmi=23.0, hba1c=5.5,
        glucose=105.0, hr=68, spo2=99, steps=10000, sleep_hours=8.0,
    ),
    RiskFeatures(
        age=58, gender="M", bmi=31.0, hba1c=8.1, has_hypertension=True,
        medication_count=2, glucose=195.0, hr=90, spo2=95, steps=3000, sleep_hours=6.0,
        thirst=True,
    ),
    RiskFeatures(
        age=72, gender="M", bmi=38.0, hba1c=11.5, has_hypertension=True,
        has_heart_disease=True, medication_count=4,
        glucose=380.0, hr=105, spo2=91, steps=800, sleep_hours=5.0,
        confusion=True, thirst=True,
    ),
]

for i, p in enumerate(patients, 1):
    pred = rc.predict(p)
    print(f"Patient {i}: risk={pred.risk_level} ({pred.confidence:.0%})")
    print(f"  top features : {pred.top_features}")
    print(f"  class probs  : {pred.class_probabilities}")
    print()